### Data Preperation

In this file, data from the Trans-Atlantic Slave Trade Database (SlaveVoyages) (https://www.slavevoyages.org/blog/the-transatlantic-slave-trade-database/163, last accessed 20.09.2025) is prepared for the spatial analysis of the slave trade.

In [1]:
#importing relevant packages
import pandas as pd
import geopandas as gp
import time
from shapely.geometry import LineString

In [2]:
#directories of input and output files
data_dir = "../data/"

In [3]:
# Loading the data into a dataframe
stdb = pd.read_spss(data_dir+'tastdb-exp-2020.sav',
                    usecols=[
    #Voyage-Info
    "VOYAGEID", #ID for identification
    "DATEDEP", #Date that voyage began
    "DATEEND", #Date when voyage completed
    "YEAR5", #5-year period in which voyage occurred
    "YEAR10", #Decade in which voyage occurred
    "YEAR25", #Quarter-century in which voyage occurred
    "YEAR100", #Century in which voyage occurred
    "NATINIMP", #Imputed country in which ship registered
    "YRCONS", #Year of vessel’s construction
    "YRREG", #Year of vessel’s registration
    "YEARDEP", #Year voyage began (imputed)
    "PLACCONS", #Place where vessel constructed
    "PLACREG", #Place where vessel registered

    #Embarkation
    "YEARAF", #Year departed Africa (imputed)
    "MJBYPTIMP", #Imputed principal place of slave purchase
    "DATEBUY", #Date that slave purchase began
    "SLAXIMP", #Imputed total slaves embarked

    #Disembarkation
    "YEARAM", #Year of arrival at port of disembarkation (imputed)
    "MJSLPTIMP", #Imputed principal place of slave disembarkation
    "SLA1PORT", #First place of slave landing
    "ADPSALE1", #Second place of slave landing
    "ADPSALE2", #Third place of slave landing
    "DATELAND1", #Date that slaves landed at first place (YYYY-MM-DD)
    "DATELAND2", #Date that slaves landed at second place
    "DATELAND3", #Date that slaves landed at third place
    "SLAMIMP", #Imputed total slaves disembarked
    "SLAARRIV" #Total slaves arrived at first port of disembarkation
             ]   
                    )

In [4]:
#change datatype of ID-column and set it as index
stdb = stdb.astype({'VOYAGEID': int})
stdb = stdb.set_index("VOYAGEID")

In [5]:
#extract categorical columns
categorical_columns = stdb.select_dtypes(include=['category']).columns

#convert categorical columns to str (geopackage cannot deal with that)
for column in categorical_columns:
    stdb[column] = stdb[column].astype(str)

In [6]:
#set date-columns as datetime
stdb[["DATEDEP","DATEEND","DATEBUY","DATELAND1","DATELAND2","DATELAND3"]] = stdb[["DATEDEP","DATEEND","DATEBUY","DATELAND1","DATELAND2","DATELAND3"]].apply(lambda x: pd.to_datetime(x,errors = 'coerce', format = '%Y-%m-%d'))

In [6]:
#delete string 'year ' from year columns
for column in ["YEAR5","YEAR10","YEAR25"]:
   stdb[column] =  stdb[column].str.replace("years ",'')

In [7]:
stdb.describe()

,SLAARRIV,SLAMIMP,SLAXIMP,YEAR100,YEARAF,YEARAM,YEARDEP,YRCONS,YRREG
count,18361.000000,34182.000000,34477.000000,36108.000000,36108.000000,36108.000000,36108.000000,6261.000000,4917.000000
mean,275.726159,269.235124,309.353134,1715.774898,1764.264983,1764.327185,1763.864296,1767.698610,1747.184259
std,159.107784,137.320017,154.727870,68.603335,59.487223,59.468134,59.519535,28.117804,56.800496
min,0.000000,0.000000,0.000000,1500.000000,1514.000000,1514.000000,1514.000000,1548.000000,1526.000000
25%,157.000000,177.000000,206.000000,1700.000000,1732.000000,1732.000000,1732.000000,1750.000000,1745.000000
50%,253.000000,261.000000,300.000000,1700.000000,1773.000000,1773.000000,1772.000000,1769.000000,1763.000000
75%,370.000000,350.000000,390.000000,1800.000000,1806.000000,1806.000000,1806.000000,1787.000000,1777.000000
max,1700.000000,1700.000000,2024.000000,1800.000000,1866.000000,1866.000000,1866.000000,1858.000000,1860.000000


In [8]:
# Convert float columns to integers
columns_to_convert = [
    "YEARDEP",
    "YRCONS",
    "YRREG",
    "YEARAF",
    "SLAXIMP",
    "YEARAM",
    "YEAR100"
    ]

for column in columns_to_convert:
    if stdb[column].isna().sum() == 0:
        stdb[column] = stdb[column].astype('int')


In [9]:
#import gazetteer for the toponym resolution
#source: https://guides.library.ucsc.edu/DS/Resources/Class-Specific/LALS194E/MappingData (last accessed 21.09.2025)
ports = pd.read_csv(data_dir+"slave-voyages-ports.csv")

In [10]:
#place-columns (needed for toponym resolution)
place_columns=[
    "PLACCONS", #Place where vessel constructed
    "PLACREG", #Place where vessel registered

    "MJBYPTIMP", #Imputed principal place of slave purchase

    "MJSLPTIMP", #Imputed principal place of slave disembarkation
    "SLA1PORT", #First place of slave landing
    "ADPSALE1", #Second place of slave landing
    "ADPSALE2" #Third place of slave landing
    ]

In [11]:
#toponym resolution for columns in place_columns with ports-dataset (ca. 83s)
start_time = time.time()
for column in place_columns:
    column_name = column
    #longitude
    stdb[column_name+'_lng'] = stdb.apply(lambda x: ports.loc[ports['port'] == x[column], 'long'].reset_index(drop=True).values[0] if x[column] in ports['port'].values else None, axis=1)
    #latitude
    stdb[column_name+'_lat'] = stdb.apply(lambda x: ports.loc[ports['port'] == x[column], 'lat'].reset_index(drop=True).values[0] if x[column] in ports['port'].values else None, axis=1)
    print(column+ " finished: "+str(round(time.time() - start_time,2))+"s")

PLACCONS finished: 8.82s
PLACREG finished: 16.67s
MJBYPTIMP finished: 40.21s
MJSLPTIMP finished: 63.76s
SLA1PORT finished: 83.59s
ADPSALE1 finished: 86.97s
ADPSALE2 finished: 89.46s


In [30]:
#Export whole df to csv
stdb.to_csv(data_dir+'stdb.csv', index=False, header=True)

unten
-> only necessary columns!!

In [12]:
#columns which every geopackage-file will need
columns_for_all = [
    #Voyage-Info
    #"VOYAGEID", #ID for identification
    "YEAR5", #5-year period in which voyage occurred
    "YEAR10", #Decade in which voyage occurred
    "YEAR25", #Quarter-century in which voyage occurred
    "YEAR100", #Century in which voyage occurred
    "NATINIMP", #Imputed country in which ship registered
    "YEARDEP", #Year voyage began (imputed)
    "YEARAM", # year of slave landing
]

In [13]:
# columns with geodata
columns_geo = [
    "PLACCONS", #Place where vessel constructed
    "PLACREG", #Place where vessel registered
    "SLA1PORT", #First place of slave landing
    "ADPSALE1", #Second place of slave landing
    "ADPSALE2" #Third place of slave landing
]

In [14]:
#specific columns for export per geo-column
columns_specific = [
    [
        "DATEDEP", #Date that voyage began
        "DATEEND", #Date when voyage completed
        "YRCONS", #Year of vessel’s construction
    ], #PLACCONS #Place where vessel constructed
    [
        "DATEDEP", #Date that voyage began
        "DATEEND", #Date when voyage completed
        "YRREG", #Year of vessel’s construction
	    "PLACCONS", #Place where vessel constructed
    ], #PLACREG #Place where vessel registered
    [
        "DATEDEP", #Date that voyage began 
        "YRCONS", #Year of vessel’s construction
        "YEARAF", #Year departed Africa (imputed)
        "MJBYPTIMP", #Imputed principal place of slave purchase
        "DATEBUY", #Date that slave purchase began
        "SLAXIMP", #Imputed total slaves embarked
    ], #PTDEPIMP #Imputed port where voyage began
    [
        "DATEEND", #Date when voyage completed 
            #Disembarkation
        "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
        "SLA1PORT", #First place of slave landing
        "DATELAND1", #Date that slaves landed at first place (YYYY-MM-DD)
        "DATELAND2", #Date that slaves landed at second place
        "DATELAND3", #Date that slaves landed at third place
        "SLAMIMP", #Imputed total slaves disembarked
        "SLAARRIV", #Total slaves arrived at first port of disembarkation
    ], #PORTRET #Port at which voyage ended
    [
        #Disembarkation
        "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
        "DATELAND1", #Date that slaves landed at first place (YYYY-MM-DD)
        "SLAARRIV", #Total slaves arrived at first port of disembarkation
    ], #SLA1PORT #First place of slave landing
    [
        #Disembarkation
        "DATELAND2", #Date that slaves landed at second place
    ], #ADPSALE1 #Second place of slave landing
    [
        #Disembarkation
        "DATELAND3", #Date that slaves landed at third place
    ] #ADPSALE2 #Third place of slave landing

]

In [45]:
#loop through columns_geo, filtered for relevant columns, export as single geopackage-files 
start_time = time.time()
for i in range(0,len(columns_geo)):
    #relevant columns
    geocolumn = columns_geo[i]
    relevant_columns = columns_for_all + columns_specific[i]
    relevant_columns.insert(0,columns_geo[i])
    relevant_columns.append(geocolumn+"_lng")
    relevant_columns.append(geocolumn+"_lat")

    #filter dataframe with relevant columns
    stdb_filter_columns = stdb[relevant_columns]

    # filter data where geometry is not Null
    stdb_filtered_geo = stdb_filter_columns[(stdb_filter_columns[[geocolumn+'_lng', geocolumn+'_lat']] != "0").all(axis=1) & stdb_filter_columns[[geocolumn+'_lng', geocolumn+'_lat']].notna().all(axis=1)]

    #dataframe to geodataframe
    gdf = gp.GeoDataFrame(stdb_filtered_geo, geometry=gp.points_from_xy(stdb_filtered_geo[geocolumn+'_lng'], stdb_filtered_geo[geocolumn+'_lat']), crs="EPSG:4326")
    
    #export gdf as geopackage
    gdf.to_file(data_dir+f"{geocolumn}.gpkg", driver="GPKG")

    print(geocolumn+ " finished: "+str(round(time.time() - start_time,2))+"s")

PLACCONS finished: 0.42s
PLACREG finished: 0.83s
SLA1PORT finished: 1.75s
ADPSALE1 finished: 2.03s
ADPSALE2 finished: 2.28s


In [ ]:
#filter data for SEL->DISEMB
columns_specific = [
    "DATEDEP", #Date that voyage began
    "DATEEND", #Date when voyage completed
    "YEARAF", #Year departed Africa (imputed)
    "MJBYPTIMP", #Imputed principal place of slave purchase
    "DATEBUY", #Date that slave purchase began
    "MJSLPTIMP", #Imputed principal place of slave disembarkation, DAP for Animation
    "SLAMIMP", #Imputed total slaves disembarked
    "MJBYPTIMP_lng",
    "MJBYPTIMP_lat",
    "MJSLPTIMP_lng",
    "MJSLPTIMP_lat"
]

#filter df for relevant columns
relevant_columns = columns_for_all + columns_specific
stdb_filtered = stdb[relevant_columns]

#export df to csv
stdb_filtered.to_csv(data_dir+'stdb_BYSEL.csv', index=False, header=True)

#apply additional filter: Remove rows where lat or lng are null
stdb_filtered2 = stdb_filtered.dropna(subset=["MJBYPTIMP_lng", "MJBYPTIMP_lat", "MJSLPTIMP_lng", "MJSLPTIMP_lat"])

#export filtered DataFrame to CSV
stdb_filtered2.to_csv(data_dir+'stdb_BYSEL_FILTER.csv', index=False, header=True)


In [49]:
#create LineString geometries from start and end coordinates
geometry = stdb_filtered2.apply(
    lambda row: LineString([
        (row["MJBYPTIMP_lng"], row["MJBYPTIMP_lat"]),  # Start point
        (row["MJSLPTIMP_lng"], row["MJSLPTIMP_lat"])   # End point
    ]), axis=1
)

#convert DataFrame to GeoDataFrame
gdf = gp.GeoDataFrame(stdb_filtered2, geometry=geometry)

#set CRS (Coordinate Reference System) to WGS 84 (EPSG:4326)
gdf.set_crs(epsg=4326, inplace=True)

#export to GeoPackage
output_path = data_dir+"stdb_lines.gpkg"
gdf.to_file(output_path, layer="lines", driver="GPKG")

print(f"GeoPackage saved at {output_path}")


GeoPackage saved at ../data/stdb_lines.gpkg


-> maybe relevant for 3rd question??

In [17]:
#file for ports-BUY
stdb_SEL= stdb[["MJBYPTIMP","MJBYPTIMP_lng","MJBYPTIMP_lat"]]

#filter data where geometry is not Null
stdb_filtered_geo = stdb_SEL[(stdb_SEL[['MJBYPTIMP_lng', 'MJBYPTIMP_lat']] != "0").all(axis=1) & stdb_SEL[['MJBYPTIMP_lng', 'MJBYPTIMP_lat']].notna().all(axis=1)]

#drop duplicate rows
stdb_filtered_geo = stdb_filtered_geo.drop_duplicates()

#dataframe to geodataframe
gdf = gp.GeoDataFrame(stdb_filtered_geo, geometry=gp.points_from_xy(stdb_filtered_geo['MJBYPTIMP_lng'], stdb_filtered_geo['MJBYPTIMP_lat']), crs="EPSG:4326")

#export gdf as geopackage
gdf.to_file(data_dir+"stdb_BUY.gpkg", driver="GPKG")

In [18]:
#file for ports-SEL
stdb_BUY= stdb[["MJSLPTIMP", "MJSLPTIMP_lng", "MJSLPTIMP_lat","YEARAM", "SLAMIMP", "NATINIMP"]]

#filter data where geometry is not Null
stdb_filtered_geo = stdb_BUY[(stdb_BUY[['MJSLPTIMP_lng', 'MJSLPTIMP_lat']] != "0").all(axis=1) & stdb_BUY[['MJSLPTIMP_lng', 'MJSLPTIMP_lat']].notna().all(axis=1)]

#drop duplicate rows
stdb_filtered_geo = stdb_filtered_geo.drop_duplicates()

#dataframe to geodataframe
gdf = gp.GeoDataFrame(stdb_filtered_geo, geometry=gp.points_from_xy(stdb_filtered_geo['MJSLPTIMP_lng'], stdb_filtered_geo['MJSLPTIMP_lat']), crs="EPSG:4326")

#export gdf as geopackage
gdf.to_file(data_dir+"stdb_SEL.gpkg", driver="GPKG")